# Demo: Red de Hopfield

Carga el dataset, entrena una Red de Hopfield, agrega ruido y visualiza el patron original, ruidoso y recuperado.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from src.models.hopfield import HopfieldNetwork


In [ ]:
def load_dataset(size: int):
    path = PROJECT_ROOT / 'dataset' / f'alphabet_{size}x{size}.npz'
    data = np.load(path, allow_pickle=False)
    return data['labels'], data['matrices'].astype(np.int8), data['vectors'].astype(np.int8)

def add_noise(pattern: np.ndarray, noise_level: float, random_state: int = 42) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    noisy = pattern.copy()
    n_flip = int(round(noise_level * noisy.size))
    idx = rng.choice(noisy.size, size=n_flip, replace=False)
    noisy[idx] *= -1
    return noisy

def plot_patterns(patterns, titles, size: int):
    plt.figure(figsize=(10, 3))
    for i, (pattern, title) in enumerate(zip(patterns, titles), start=1):
        plt.subplot(1, 3, i)
        plt.imshow(pattern.reshape(size, size), cmap='gray_r', interpolation='nearest')
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
size = 10
labels, matrices, vectors = load_dataset(size)

print('labels:', labels.shape)
print('matrices:', matrices.shape)
print('vectors:', vectors.shape)


In [ ]:
n_patterns = 5
network = HopfieldNetwork(n_neurons=size * size)
network.train(vectors[:n_patterns])
print('Letras almacenadas:', ''.join(labels[:n_patterns].astype(str)))


In [ ]:
letter_index = 0
noise_level = 0.10

original = vectors[letter_index]
noisy = add_noise(original, noise_level=noise_level, random_state=7)
recovered = network.recall(noisy, max_iter=100)

print('Letra:', labels[letter_index])
print('Convergio:', network.converged)
print('Iteraciones:', network.last_iterations)
print('Recuperacion exacta:', np.array_equal(original, recovered))


In [ ]:
plot_patterns(
    [original, noisy, recovered],
    ['Original', f'Ruidoso {int(noise_level * 100)}%', 'Recuperado'],
    size=size,
)
